# B1-H2 original test inference — Colab L4
Select Runtime Version **2026.10**, GPU **L4**, and run all cells. Mount the same Drive account on every session.
The notebook downloads its pinned input bundle automatically. Thresholds are fixed at Devign 0.09 and ReVeal 0.67.
This is the disclosed Colab platform deviation from preregistered Kaggle 2×T4.

Completed samples are saved individually on Drive. After disconnection, reconnect and run from the top:
matching checkpoint records are skipped automatically; an interrupted in-flight sample is scored again.
No artificial activity/keepalive is used. The session stops safely before its 9-hour budget.
Full setup/native logs remain in output files; progress is printed every 50 records.
Inference refuses to start unless the native log verifies every model layer was offloaded.
Bundle: https://drive.google.com/file/d/1fd8sscUUmeoSbiiwB7SHWeljytYwHMee/view


In [1]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, subprocess, platform, time, re, shutil, math, contextlib, io, importlib.metadata
from datetime import datetime, timezone

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f'Requires Colab Runtime Version 2026.10 / Python 3.13.x to match the Kaggle Python 3.13 runtime family; got {sys.version.split()[0]}. In Runtime → Change runtime type choose Runtime Version 2026.10, then reconnect and run from the top.')
_nvidia_smi = shutil.which('nvidia-smi') or ('/usr/local/cuda/bin/nvidia-smi' if Path('/usr/local/cuda/bin/nvidia-smi').is_file() else None)
if _nvidia_smi:
    _preflight_gpu = subprocess.run([_nvidia_smi, '--query-gpu=name', '--format=csv,noheader'], check=True, text=True, capture_output=True).stdout.strip()
else:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError('Cannot detect GPU: nvidia-smi is missing and PyTorch is unavailable. Select Runtime → Change runtime type → GPU L4, reconnect, then run again.') from exc
    if not torch.cuda.is_available():
        raise RuntimeError('Colab has not attached a CUDA GPU (nvidia-smi missing and torch.cuda.is_available() is False). Select GPU L4 in Runtime → Change runtime type, reconnect, then run again.')
    _preflight_gpu = torch.cuda.get_device_name(0)
if 'L4' not in _preflight_gpu.upper():
    raise RuntimeError(f'This pilot is configured for L4; selected runtime reports {_preflight_gpu!r}. Select GPU L4 and reconnect.')
print(f'Preflight passed: Python {sys.version.split()[0]}, GPU {_preflight_gpu}')

WORK = Path('/content/b1h2_l4_test')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path('/content/drive/MyDrive/GRACE_B1H2_COLAB_L4_TEST_20261106')
BUNDLE_DRIVE_URL = 'https://drive.google.com/file/d/1fd8sscUUmeoSbiiwB7SHWeljytYwHMee/view'
BUNDLE_PATH = DRIVE_DIR / 'b1h2_original_test_bundle.zip'
OUT = DRIVE_DIR / 'l4_test_output'
EVENTS = OUT / 'events.log'
PILOT_START = time.monotonic()
MAX_RUNTIME_SECONDS = 9 * 60 * 60

def note(msg):
    line = f'[{datetime.now(timezone.utc).isoformat()}] {msg}'
    print(line, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f: f.write(line + '\n')

BUNDLE_EXPECTED_SHA256 = 'c1557cdd26c2e59a5e2f77fee42cd94f6dc802558fbdd5a7745facc791257c24'
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
if not BUNDLE_PATH.is_file():
    from urllib.request import urlopen
    # Download to local disk, then publish to Drive only after SHA verification.
    local_bundle = WORK / BUNDLE_PATH.name
    with urlopen('https://drive.google.com/uc?export=download&id=' + BUNDLE_DRIVE_URL.split('/d/')[1].split('/')[0]) as response:
        local_bundle.write_bytes(response.read())
    if hashlib.sha256(local_bundle.read_bytes()).hexdigest() != BUNDLE_EXPECTED_SHA256:
        raise ValueError('Downloaded bundle hash mismatch; place the bundle from the link in DRIVE_DIR manually.')
    shutil.copy2(local_bundle, BUNDLE_PATH)
note(f'Bundle: {BUNDLE_PATH}; output: {OUT}')

LOCAL_OUT = WORK / 'session_logs'
LOCAL_OUT.mkdir(exist_ok=True)
SESSION_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+str(time.time_ns())
SESSION_LOG_DIR=OUT/'sessions'/SESSION_ID
SESSION_LOG_DIR.mkdir(parents=True,exist_ok=True)
CHECKPOINTS = OUT / 'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
def atomic_json(path, value):
    path = Path(path)
    raw = (json.dumps(value, ensure_ascii=False, allow_nan=False) + '\n').encode()
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as f:
        f.write(raw); f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)
    if path.read_bytes() != raw: raise OSError('Drive checkpoint readback failed: '+str(path))
def note(msg, display=True):
    line=f'[{datetime.now(timezone.utc).isoformat()}] {msg}'
    with (LOCAL_OUT/'events.log').open('a',encoding='utf-8') as f: f.write(line+'\n')
    if display: print(line,flush=True)


Preflight passed: Python 3.13.16, GPU NVIDIA L4
Mounted at /content/drive
[2026-11-06T09:15:10.851496+00:00] Bundle: /content/drive/MyDrive/GRACE_B1H2_COLAB_L4_TEST_20261106/b1h2_original_test_bundle.zip; output: /content/drive/MyDrive/GRACE_B1H2_COLAB_L4_TEST_20261106/l4_test_output


In [2]:
# Record runtime facts; nvidia-smi is optional if torch can identify the assigned GPU.
_nvidia_smi = shutil.which('nvidia-smi') or ('/usr/local/cuda/bin/nvidia-smi' if Path('/usr/local/cuda/bin/nvidia-smi').is_file() else None)
if _nvidia_smi:
    gpu = subprocess.run([_nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True, text=True, capture_output=True).stdout.strip()
    (SESSION_LOG_DIR / 'runtime_nvidia_smi.txt').write_text(subprocess.run([_nvidia_smi], check=True, text=True, capture_output=True).stdout, encoding='utf-8')
else:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU unavailable; reconnect with GPU L4 selected.')
    props = torch.cuda.get_device_properties(0)
    gpu = f'{torch.cuda.get_device_name(0)}, {props.total_memory / (1024**2):.0f} MiB; nvidia-smi unavailable'
    (SESSION_LOG_DIR / 'runtime_nvidia_smi.txt').write_text(gpu + '\n', encoding='utf-8')
if 'L4' not in gpu.upper(): raise RuntimeError(f'Expected L4, got {gpu!r}; select GPU L4 and reconnect.')
print(gpu)
_nvcc = shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if Path('/usr/local/cuda/bin/nvcc').is_file() else None)
if not _nvcc: raise RuntimeError('CUDA GPU found but nvcc is missing. Reconnect to a Colab GPU runtime with the CUDA toolkit, then rerun.')
for cmd, name in [([_nvcc, '--version'], 'nvcc_version.txt'), ([sys.executable, '--version'], 'python_version.txt')]:
    p = subprocess.run(cmd, text=True, capture_output=True)
    (SESSION_LOG_DIR / name).write_text(p.stdout + p.stderr, encoding='utf-8')
try:
    memory_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
except (AttributeError, OSError, ValueError):
    memory_gib = None
if memory_gib is not None:
    (SESSION_LOG_DIR / 'system_memory_gib.txt').write_text(f'{memory_gib:.2f} GiB\n', encoding='utf-8')
    print(f'System RAM: {memory_gib:.2f} GiB')
note(f'Confirmed Python {sys.version.split()[0]} and L4 runtime; GPU compute capability for this build is SM 8.9.')


NVIDIA L4, 23034 MiB, 580.82.07
System RAM: 52.96 GiB
[2026-11-06T09:15:11.209857+00:00] Confirmed Python 3.13.16 and L4 runtime; GPU compute capability for this build is SM 8.9.


In [3]:
# Pin dependencies and build llama.cpp with logs on Colab's local disk.
# Google Drive FUSE can disconnect during a long CUDA build, so never stream compiler output to OUT.
LOCAL_LOG_DIR = Path('/content/b1h2_l4_setup_logs')
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

def sync_setup_log(name):
    source = LOCAL_LOG_DIR / name
    try:
        OUT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, SESSION_LOG_DIR / name)
        print(f'Saved setup log to Drive: {SESSION_LOG_DIR / name}', flush=True)
    except OSError as exc:
        print(f'Could not copy {name} to Drive ({exc}); local copy remains at {source}', flush=True)

packages = [
    'numpy==2.1.3',
    'huggingface-hub==0.36.2',
    'tokenizers==0.22.2',
    'transformers==4.57.1',
    'scikit-learn==1.5.2',
]
name = 'install_python_dependencies.log'
with (LOCAL_LOG_DIR / name).open('w', encoding='utf-8') as log:
    p = subprocess.Popen([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    line_count = 0
    for line in p.stdout:
        log.write(line); line_count += 1
        if line_count % 500 == 0: print(f'Python dependencies: captured {line_count} lines', flush=True)
    result = p.wait()
sync_setup_log(name)
if result != 0: raise RuntimeError(f'Pinned Python dependency installation failed; local log: {LOCAL_LOG_DIR / name}')

os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=89'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
name = 'install_llama_cpp_cuda.log'
with (LOCAL_LOG_DIR / name).open('w', encoding='utf-8') as log:
    cmd = [sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary', 'llama-cpp-python', 'llama-cpp-python==0.3.16']
    log.write(f'Command: {cmd}\nCMAKE_ARGS={os.environ["CMAKE_ARGS"]}\n'); log.flush()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
    line_count = 0
    for line in p.stdout:
        log.write(line); line_count += 1
        if line_count % 2000 == 0: print(f'CUDA build: captured {line_count} lines', flush=True)
    result = p.wait()
sync_setup_log(name)
if result != 0: raise RuntimeError(f'CUDA build of llama-cpp-python failed; local log: {LOCAL_LOG_DIR / name}')


Saved setup log to Drive: /content/drive/MyDrive/GRACE_B1H2_COLAB_L4_TEST_20261106/l4_test_output/sessions/20261106T091510Z_1793956510858021740/install_python_dependencies.log
CUDA build: captured 2000 lines
Saved setup log to Drive: /content/drive/MyDrive/GRACE_B1H2_COLAB_L4_TEST_20261106/l4_test_output/sessions/20261106T091510Z_1793956510858021740/install_llama_cpp_cuda.log


In [4]:
# Verify archive, source prompts, fixed sample IDs, and registered checkpoint identity.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''): h.update(b)
    return h.hexdigest()

with zipfile.ZipFile(BUNDLE_PATH) as z:
    bundle_manifest = json.loads(z.read('bundle_manifest.json'))
    prompt_bytes = z.read('prompts.jsonl')
if hashlib.sha256(prompt_bytes).hexdigest() != bundle_manifest['prompts_sha256']:
    raise ValueError('Prompt bundle hash mismatch')
rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
if len(rows) != bundle_manifest['count'] or len({r['id'] for r in rows}) != bundle_manifest['count']:
    raise ValueError('Expected complete unique test prompt IDs')
if bundle_manifest['source_archive_sha256'] != '9db0d3eebb66a2f83c1f9803ceb0192954a3dfe2c3de9515188a996ede84ed7b':
    raise ValueError('Unexpected source archive hash')
if bundle_manifest['prompts_sha256'] != 'd770d1d28c58c8c0d32d7b9e0cfc1b29820332969b4a0edba78eb33ea1f0f5fb':
    raise ValueError('Unexpected prompt hash')
for row in rows:
    if hashlib.sha256(row['prompt'].encode('utf-8')).hexdigest() != row['prompt_sha256']:
        raise ValueError(f'Prompt hash mismatch: {row["id"]}')
if {d: sum(r['dataset'] == d for r in rows) for d in bundle_manifest['dataset_counts']} != bundle_manifest['dataset_counts']:
    raise ValueError('Test dataset counts differ from manifest')
note(f'Bundle verified: {len(rows)} prompts; prompts_sha256={bundle_manifest["prompts_sha256"]}')

if sha256_file(BUNDLE_PATH) != BUNDLE_EXPECTED_SHA256: raise ValueError('Bundle archive changed')
with zipfile.ZipFile(BUNDLE_PATH) as z: locked_thresholds=json.loads(z.read('thresholds.json'))
seal=locked_thresholds.pop('locked_sha256')
computed=hashlib.sha256(json.dumps(locked_thresholds,indent=2,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
if seal != computed or seal != bundle_manifest['threshold_lock_sha256']: raise ValueError('Threshold seal mismatch')
THRESHOLDS=bundle_manifest['thresholds']
for ds in ('devign','reveal'):
    if THRESHOLDS[ds] != locked_thresholds['thresholds'][f'{ds}|B1-H2|42']: raise ValueError('Threshold mismatch')
for row in rows:
    if row['split']!='test' or row['arm']!='B1-H2' or row['seed']!=42: raise ValueError('Wrong input arm/split/seed')
RUN_IDENTITY=dict(model_sha256=bundle_manifest.get('model_sha256','f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'),
                  prompts_sha256=bundle_manifest['prompts_sha256'],threshold_lock_sha256=seal,arm='B1-H2',split='test',seed=42)
identity_path=OUT/'checkpoint_identity.json'
if identity_path.exists() and json.loads(identity_path.read_text()) != RUN_IDENTITY:
    raise ValueError('Existing checkpoint belongs to different input/model/thresholds')
atomic_json(identity_path,RUN_IDENTITY)


[2026-11-06T09:33:16.775479+00:00] Bundle verified: 5006 prompts; prompts_sha256=d770d1d28c58c8c0d32d7b9e0cfc1b29820332969b4a0edba78eb33ea1f0f5fb


In [5]:
# Download the registered GGUF and refuse a changed artifact.
from huggingface_hub import hf_hub_download
MODEL_REPO = 'ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF'
MODEL_FILE = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
TOKENIZER_REPO = 'Qwen/Qwen3-Coder-30B-A3B-Instruct'
TOKENIZER_REVISION = 'b2cff646eb4bb1d68355c01b18ae02e7cf42d120'
model_path = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision='main'))
model_digest = sha256_file(model_path)
if model_digest != MODEL_SHA256:
    raise ValueError(f'GGUF SHA-256 mismatch: expected {MODEL_SHA256}, got {model_digest}')
note(f'Model verified: {model_path}; sha256={model_digest}; revision=local-sha256:{model_digest}')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/18.0G [00:00<?, ?B/s]

[2026-11-06T09:35:34.307029+00:00] Model verified: /root/.cache/huggingface/hub/models--ggml-org--Qwen3-Coder-30B-A3B-Instruct-GGUF/snapshots/0ca51122012fe4a20dc82988ee58780b008ce4e3/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf; sha256=f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335; revision=local-sha256:f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335


In [6]:
# Capture the llama.cpp callback as well as native file descriptors, on LOCAL disk.
# Python redirect_stderr alone does not capture the llama.cpp C callback.
from transformers import AutoTokenizer
from llama_cpp import Llama, __version__ as llama_cpp_version, llama_supports_gpu_offload
from llama_cpp import llama_cpp as native
if llama_cpp_version != '0.3.16' or not llama_supports_gpu_offload(): raise RuntimeError('Pinned CUDA backend unavailable')
tokenizer=AutoTokenizer.from_pretrained(TOKENIZER_REPO,revision=TOKENIZER_REVISION)
label_ids=[tokenizer.encode(x,add_special_tokens=False)[0] for x in ('0','1')]
if any(len(tokenizer.encode(x,add_special_tokens=False))!=1 for x in ('0','1')): raise ValueError('Labels must be single tokens')
native_path=LOCAL_OUT/'llama_cpp_model_load_offload.log'
native_file=native_path.open('wb',buffering=0)
@native.llama_log_callback
def native_log_callback(level, text, user_data):
    try:
        if text: os.write(native_file.fileno(),text)
    except Exception: pass
# Keep callback and file alive for the lifetime of the model.
native.llama_log_set(native_log_callback,None)
saved=[os.dup(1),os.dup(2)]
try:
    sys.stdout.flush(); sys.stderr.flush()
    os.dup2(native_file.fileno(),1); os.dup2(native_file.fileno(),2)
    llm=Llama(model_path=str(model_path),n_ctx=8192,n_batch=256,n_gpu_layers=-1,logits_all=True,seed=42,verbose=True)
finally:
    sys.stdout.flush(); sys.stderr.flush()
    os.dup2(saved[0],1); os.dup2(saved[1],2)
    os.close(saved[0]); os.close(saved[1])
    os.fsync(native_file.fileno())
    shutil.copy2(native_path,SESSION_LOG_DIR/native_path.name)
load_log=native_path.read_text(encoding='utf-8',errors='replace')
layer_reports=re.findall(r'offloaded\s+(\d+)\s*/\s*(\d+)\s+layers?\s+to\s+GPU',load_log,re.I)
all_layers_offloaded=bool(layer_reports) and any(int(a)==int(b) and int(b)>0 for a,b in layer_reports)
post_gpu=subprocess.run([_nvidia_smi],text=True,capture_output=True,check=True).stdout
(SESSION_LOG_DIR/'nvidia_smi_after_model_load.txt').write_text(post_gpu,encoding='utf-8')
summary=dict(status='loaded',gpu=gpu,all_layers_offloaded=all_layers_offloaded,layer_reports=layer_reports,
             model_sha256=model_digest,python=sys.version,packages={p:importlib.metadata.version(p) for p in
             ('llama-cpp-python','transformers','huggingface-hub','tokenizers','numpy','scikit-learn')},
             context_tokens=8192,n_batch=256,n_gpu_layers=-1,native_log_bytes=native_path.stat().st_size)
atomic_json(SESSION_LOG_DIR/'load_summary.json',summary)
if not all_layers_offloaded:
    raise RuntimeError('Test blocked: full GPU offload was not verified. Inspect native log and post-load nvidia-smi; do not infer or silently change GPU.')
for label,hf_id in zip(('0','1'),label_ids):
    if llm.tokenize(label.encode(),add_bos=False,special=False)!=[hf_id]: raise ValueError('Label tokenizer mismatch')
llm.verbose=False
note(f'Offload verified: {layer_reports}; full log saved to Drive.')


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

CUDA : ARCHS = 890 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | AVX512_VNNI = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'general.quantization_version': '2', 'tokenizer.ggml.normalizer.nfc': 'true', 'tokenizer.ggml.add_eos_token': 'false', 'general.file_type': '15', 'qwen3moe.context_length': '262144', 'general.type': 'model', 'tokenizer.ggml.clean_spaces': 'false', 'qwen3moe.rope.freq_base': '10000000.000000', 'tokenizer.ggml.bos_token_id': '151643', 'general.name': 'Qwen3-Coder-30B-A3B-Instruct', 'qwen3moe.attention.layer_norm_rms_epsilon': '0.000001', 'qwen3moe.attention.key_length': '128', 'qwen3moe.embedding_length': '2048', 'tokenizer.ggml.padding_token_id': '151643', 'general.license': 'apache-2.0', 'qwen3moe.feed_forward_length': '6144', 'general.sampling.temp': '0.700000', 'general.sampling.top_p': '0.800000', 'general.sampling.penalty_repeat': '1.050000', 'gen

[2026-11-06T09:36:27.129936+00:00] Offload verified: [('49', '49')]; full log saved to Drive.


In [7]:
# Resume-safe inference over all remaining validation prompts; one record is fsynced after each completed prompt.
import numpy as np

def binary_scores(logits, ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    m = float(np.max(values))
    normalizer = m + float(np.log(np.exp(values - m).sum()))
    lp0 = float(values[ids[0]] - normalizer)
    lp1 = float(values[ids[1]] - normalizer)
    s = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    top = np.argsort(values)[-5:][::-1]
    return {
        'logprob_0': lp0, 'logprob_1': lp1,
        'p0': math.exp(lp0), 'p1': math.exp(lp1), 's': s,
        'pred_argmax': int(s >= 0.5),
        'greedy_token_id': int(np.argmax(values)),
        'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top],
    }


expected={r['id']:r for r in rows}
done={}
def checkpoint_path(uid): return CHECKPOINTS/(hashlib.sha256(uid.encode()).hexdigest()+'.json')
def validate_record(r,item):
    for key in ('id','dataset','split','arm','seed','prompt_sha256'):
        if r.get(key)!=item[key]: raise ValueError('Checkpoint identity mismatch: '+item['id']+' '+key)
    if r.get('model_sha256')!=model_digest or r.get('threshold_lock_sha256')!=seal:
        raise ValueError('Checkpoint model/threshold seal mismatch: '+item['id'])
    if r.get('tau')!=THRESHOLDS[item['dataset']] or not r.get('all_layers_offloaded'):
        raise ValueError('Checkpoint threshold/offload mismatch: '+item['id'])
    if not all(math.isfinite(r[k]) for k in ('s','logprob_0','logprob_1')): raise ValueError('Invalid checkpoint scores')
available_checkpoints={p.name:p for p in CHECKPOINTS.glob('*.json')}
for uid,item in expected.items():
    p=checkpoint_path(uid)
    if p.name not in available_checkpoints: continue
    try: r=json.loads(p.read_text(encoding='utf-8'))
    except (json.JSONDecodeError,UnicodeDecodeError):
        quarantine=p.with_suffix('.corrupt.'+str(time.time_ns()))
        os.replace(p,quarantine)
        note('Quarantined incomplete checkpoint: '+uid)
        continue
    validate_record(r,item); done[uid]=r
note(f'Resume: {len(done)}/{len(rows)} saved; {len(rows)-len(done)} remaining')
durations=[]
try:
    for item in rows:
        if item['id'] in done: continue
        reserve=max(120,2*max(durations,default=60))
        if time.monotonic()-PILOT_START+reserve>=MAX_RUNTIME_SECONDS:
            note('Safety budget reached; rerun from top in next L4 session to resume.'); break
        chat=tokenizer.apply_chat_template([{'role':'user','content':item['prompt']}],tokenize=False,add_generation_prompt=True)
        hf_ids=tokenizer.apply_chat_template([{'role':'user','content':item['prompt']}],tokenize=True,add_generation_prompt=True)
        gguf_ids=llm.tokenize(chat.encode(),add_bos=False,special=True)
        if list(hf_ids)!=gguf_ids or len(gguf_ids)>8192: raise ValueError('Tokenizer/context mismatch: '+item['id'])
        started=time.monotonic(); llm.reset(); llm.eval(gguf_ids)
        scores=binary_scores(llm.scores[llm.n_tokens-1],label_ids)
        raw=llm.detokenize([scores['greedy_token_id']]).decode('utf-8',errors='replace').strip()
        tau=THRESHOLDS[item['dataset']]
        r={k:item[k] for k in ('id','dataset','split','arm','seed','prompt_variant','prompt_sha256')}
        r.update(scores,raw=raw,response_valid=raw in ('0','1'),valid=True,tau=tau,pred_tau=int(scores['s']>=tau),
                 threshold_lock_sha256=seal,model_sha256=model_digest,tokenizer_revision=TOKENIZER_REVISION,
                 backend='llama-cpp-python',llama_cpp_python=llama_cpp_version,gpu=gpu,all_layers_offloaded=True,
                 prompt_tokens=len(gguf_ids),token_ids={'0':label_ids[0],'1':label_ids[1]},elapsed_s=time.monotonic()-started,
                 completed_utc=datetime.now(timezone.utc).isoformat(),test_only=True)
        # Retain a local pending copy if Drive fails. Stop before evaluating another sample.
        atomic_json(LOCAL_OUT/'pending_record.json',r)
        atomic_json(checkpoint_path(item['id']),r)
        validate_record(json.loads(checkpoint_path(item['id']).read_text()),item)
        done[item['id']]=r; durations.append(r['elapsed_s'])
        note(f'{item["id"]}: s={scores["s"]:.8f}, pred_tau={r["pred_tau"]}, seconds={r["elapsed_s"]:.2f}',display=False)
        if len(done)%50==0:
            note(f'Progress {len(done)}/{len(rows)}; remaining {len(rows)-len(done)}')
            shutil.copy2(LOCAL_OUT/'events.log',SESSION_LOG_DIR/'events.log')
            os.fsync(native_file.fileno()); shutil.copy2(native_path,SESSION_LOG_DIR/native_path.name)
finally:
    # Authoritative per-ID checkpoints survive even when an interrupted session cannot reach here.
    local_records=LOCAL_OUT/'records.jsonl'
    with local_records.open('w',encoding='utf-8') as f:
        for uid in sorted(done): f.write(json.dumps(done[uid],ensure_ascii=False,allow_nan=False)+'\n')
    target=OUT/'records.jsonl'; temp=OUT/'records.jsonl.tmp'
    shutil.copy2(local_records,temp); os.replace(temp,target)
    atomic_json(OUT/'test_status.json',dict(status='complete' if len(done)==len(rows) else 'incomplete',
                expected=len(rows),records_written=len(done),thresholds=THRESHOLDS,threshold_lock_sha256=seal,
                all_layers_offloaded=all_layers_offloaded,runtime_cap_seconds=MAX_RUNTIME_SECONDS))
    os.fsync(native_file.fileno()); shutil.copy2(native_path,SESSION_LOG_DIR/native_path.name)
    shutil.copy2(LOCAL_OUT/'events.log',SESSION_LOG_DIR/'events.log')
    archive=shutil.make_archive(str(WORK/'b1h2_test_output'),'zip',root_dir=OUT)
    shutil.copy2(archive,DRIVE_DIR/'b1h2_test_output.zip')
    note(f'Preserved {len(done)}/{len(rows)} records; Drive archive: {DRIVE_DIR / "b1h2_test_output.zip"}')


[2026-11-06T09:36:27.196837+00:00] Resume: 0/5006 saved; 5006 remaining
[2026-11-06T09:37:09.608967+00:00] Progress 50/5006; remaining 4956
[2026-11-06T09:37:48.373662+00:00] Progress 100/5006; remaining 4906
[2026-11-06T09:38:30.399099+00:00] Progress 150/5006; remaining 4856
[2026-11-06T09:39:12.844584+00:00] Progress 200/5006; remaining 4806
[2026-11-06T09:39:58.407064+00:00] Progress 250/5006; remaining 4756
[2026-11-06T09:40:38.894050+00:00] Progress 300/5006; remaining 4706
[2026-11-06T09:41:21.994975+00:00] Progress 350/5006; remaining 4656
[2026-11-06T09:42:05.129398+00:00] Progress 400/5006; remaining 4606
[2026-11-06T09:42:43.289194+00:00] Progress 450/5006; remaining 4556
[2026-11-06T09:43:25.361416+00:00] Progress 500/5006; remaining 4506
[2026-11-06T09:44:13.389805+00:00] Progress 550/5006; remaining 4456
[2026-11-06T09:44:55.391773+00:00] Progress 600/5006; remaining 4406
[2026-11-06T09:45:39.336696+00:00] Progress 650/5006; remaining 4356
[2026-11-06T09:46:23.308937+00:0